# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Get metadata as a dict (to print name/description info)
meta_dict = json.loads(dataset.metadata.to_json())
print(f"{meta_dict['name']}: {meta_dict['description']}")

# Show keywords
keywords = meta_dict.get('keywords', [])
if keywords:
    print("Keywords:", ', '.join(keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve all record sets by their '@id'
all_record_sets = dataset.record_sets
print(f"Found {len(all_record_sets)} record set(s):\n")
for rs in all_record_sets:
    print(f"RecordSet: @id={rs.id}, name={rs.name}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field: @id={field.id}, name={field.name}, data_type={field.data_type}")
    print()
# We'll use the first record set for demonstration, but you can select any by @id below.
if all_record_sets:
    example_record_set_id = all_record_sets[0].id  # Use for the following steps
    print(f"Example record set selected: {example_record_set_id}")
else:
    raise RuntimeError("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Get all record set IDs for extraction as list
record_sets = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - {len(df)} records loaded. Columns: {df.columns.tolist()}")
    except Exception as e:
        print(f" - ERROR: {e}")

# Example: show DataFrame head for the first record set
main_record_set = record_sets[0]
print('\nColumns:', dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Find a numeric field from the main record set

main_record_set_obj = None
for rs in dataset.record_sets:
    if rs.id == main_record_set:
        main_record_set_obj = rs
        break
numeric_field_id = None
group_field_id = None
if main_record_set_obj is not None:
    for field in main_record_set_obj.fields:
        # Try to find a numeric field
        if field.data_type in ('Float', 'Integer', 'Number'):
            numeric_field_id = field.id
            print(f"Numeric field candidate: @id={numeric_field_id}, name={field.name}, data_type={field.data_type}")
            break
    # For grouping, pick a different non-numeric field
    for field in main_record_set_obj.fields:
        if field.data_type == 'Text' and field.id != numeric_field_id:
            group_field_id = field.id
            print(f"Grouping field candidate: @id={group_field_id}, name={field.name}")
            break
if numeric_field_id is None:
    raise RuntimeError("No numeric field found for EDA.")

# Use column ids as they appear in the DataFrame
df = dataframes[main_record_set]
numeric_field_col = numeric_field_id  # Assuming field @id used as DataFrame col name
group_field_col = group_field_id

# Try converting the numeric field to float (some Croissant data loads as strings)
df[numeric_field_col] = pd.to_numeric(df[numeric_field_col], errors='coerce')

# Remove missing values
filtered_df = df[df[numeric_field_col].notnull()]

# Threshold for example (10, or lower if the field is always below that)
threshold = filtered_df[numeric_field_col].quantile(0.5)  # Use median as a dynamic threshold
filtered_df = filtered_df[filtered_df[numeric_field_col] > threshold]
print(f"Filtered records with {numeric_field_col} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric column
field_norm = f"{numeric_field_col}_normalized"
filtered_df[field_norm] = (filtered_df[numeric_field_col] - filtered_df[numeric_field_col].mean()) / filtered_df[numeric_field_col].std()
print(f"\nNormalized {numeric_field_col} for filtered records:")
print(filtered_df[[numeric_field_col, field_norm]].head())

# Grouping
if group_field_col is not None and group_field_col in df.columns:
    grouped_df = filtered_df.groupby(group_field_col)[numeric_field_col].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_col} by {group_field_col}:")
    print(grouped_df.head())


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Histogram of the normalized numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[field_norm], kde=True, bins=30)
plt.title(f'Distribution of normalized {numeric_field_col}')
plt.xlabel(f'{numeric_field_col} (normalized)')
plt.ylabel('Count')
plt.show()

# Boxplot by grouping field (if available)
if group_field_col is not None and group_field_col in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_col, y=numeric_field_col, data=filtered_df)
    plt.title(f'{numeric_field_col} by {group_field_col}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


- This notebook demonstrated how to load, explore, and analyze the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`, referencing all entities via their `@id` for maximum reproducibility.
- Key numeric and grouping columns were dynamically selected for EDA, including normalization and simple aggregation.
- Further customized analysis and visualization can be performed by refining field selection or expanding to additional record sets as needed.
